In [1]:
import pandas as pd
import nltk
import string
import random
import pickle

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.classify import NaiveBayesClassifier
from nltk.classify.util import accuracy

CARGA DE DATOS DE ENTRENAMIENTO

In [3]:
df = pd.read_excel(r"Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\Autos\analisis_auto.xlsx")

In [4]:
columnas_necesarias = [
    "ES - General Claim - File Number",
    "texto_norm",
    "sentimiento",
    "confianza"
]

df = df[columnas_necesarias]

import re

def texto_valido(texto):
    if not isinstance(texto, str):
        return False
    texto = texto.strip()
    if texto == "":
        return False
    if len(texto) < 3:
        return False
    # comprobar que tenga al menos una letra
    if not re.search(r"[a-zA-ZáéíóúÁÉÍÓÚñÑ]", texto):
        return False
    return True

df = df[df["texto_norm"].apply(texto_valido)]

df["confianza"] = pd.to_numeric(df["confianza"], errors="coerce")
df = df[df["confianza"] >= 0.95]

print(df.head)
print(df.shape)

<bound method NDFrame.head of         ES - General Claim - File Number  \
0                              522974731   
1                              523058406   
2                              522771969   
4                              520865915   
5                              522343030   
...                                  ...   
103628                         522764577   
103629                         522747803   
103630                         522257801   
103631                         523003489   
103632                         523039312   

                                               texto_norm sentimiento  \
0          buena atención y rapidez en todos los sentidos    Positivo   
1       me repararon la luna media hora después de not...    Positivo   
2       lleve el coche para cambiar el espero derecho(...    Positivo   
4                                          se tardo mucho    Negativo   
5                ha sido todo rápido y muy buena atencion    Positivo   
...

PREPROCESAMIENTO DEL TEXTO

In [5]:
stop_words = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    texto = texto.lower()
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(texto)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

df["tokens_limpios"] = df["texto_norm"].apply(limpiar_texto)
df = df[df["tokens_limpios"].str.len() > 0]

ENTRENAMIENTO

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# X e y
X = df["texto_norm"]   
y = df["sentimiento"]   

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),   # añade trigramas
    min_df=3,             # más vocabulario
    max_df=0.85,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Logistic Regression
model = LogisticRegression(
    max_iter=2000,
    C=2.0,
    class_weight="balanced",
    solver="liblinear"
)

model.fit(X_train_tfidf, y_train)

# Evaluación
y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8942609115022908


VALIDACION CRUZADA

In [7]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    X_train_tfidf,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Accuracy por fold:", scores)
print("Accuracy medio CV:", scores.mean())
print("Desviación:", scores.std())

Accuracy por fold: [0.89156808 0.89208742 0.9        0.89751319 0.89773926]
Accuracy medio CV: 0.8957815890266649
Desviación: 0.0033474312015475743


GUARDAR MODELO

In [8]:
import joblib

joblib.dump(model, "modelo_sentimiento_autos_logreg.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer_autos.pkl")

['tfidf_vectorizer_autos.pkl']

CARGAR MODELO

In [ ]:
model = joblib.load("modelo_sentimiento_autos_logreg.pkl")
vectoriZer = joblib.load("tfidf_vectorizer_autos.pkl")